# **Inspect ROBUST04 Index - Find Available Fields**

Let's see what fields actually exist in the documents.

In [ ]:
from pyserini.search.lucene import LuceneSearcher

searcher = LuceneSearcher.from_prebuilt_index('robust04')
print("✅ Searcher loaded")

In [ ]:
# Search for a document
hits = searcher.search("international organized crime", k=1)
docid = hits[0].docid

print(f"Inspecting document: {docid}\n")

doc = searcher.doc(docid)

# Check all available methods
print("Available methods on doc object:")
methods = [m for m in dir(doc) if not m.startswith('_')]
for m in methods:
    print(f"  - {m}")

print("\n" + "="*80)
print("Testing each method:\n")

# Test docid()
print("1. doc.docid():")
try:
    result = doc.docid()
    print(f"   ✅ {result}")
except Exception as e:
    print(f"   ❌ {e}")

# Test contents()
print("\n2. doc.contents():")
try:
    result = doc.contents()
    if result:
        print(f"   ✅ {len(result)} chars: {result[:200]}...")
    else:
        print(f"   ⚠️ Returns None")
except Exception as e:
    print(f"   ❌ {e}")

# Test raw()
print("\n3. doc.raw():")
try:
    result = doc.raw()
    if result:
        print(f"   ✅ {len(result)} chars: {result[:200]}...")
    else:
        print(f"   ⚠️ Returns None")
except Exception as e:
    print(f"   ❌ {e}")

# Test lucene_document()
print("\n4. doc.lucene_document():")
try:
    lucene_doc = doc.lucene_document()
    if lucene_doc:
        print(f"   ✅ Got Lucene document object")
        print(f"   Type: {type(lucene_doc)}")
        
        # Try to get field names
        print("\n   Available fields:")
        # In Lucene, we need to iterate through fields
        try:
            # Get all field names
            from org.apache.lucene.index import IndexableField
            fields = lucene_doc.getFields()
            for field in fields:
                field_name = field.name()
                field_value = field.stringValue()
                if field_value:
                    preview = field_value[:100] if len(field_value) > 100 else field_value
                    print(f"     - {field_name}: {preview}...")
                else:
                    print(f"     - {field_name}: (no string value)")
        except Exception as e2:
            print(f"     ❌ Can't iterate fields: {e2}")
    else:
        print(f"   ⚠️ Returns None")
except Exception as e:
    print(f"   ❌ {e}")

# Try getting specific field
print("\n5. Trying specific field names:")
field_names_to_try = ['contents', 'text', 'body', 'content', 'document', 'TEXT', 'BODY']
for field_name in field_names_to_try:
    try:
        lucene_doc = doc.lucene_document()
        if lucene_doc:
            field_value = lucene_doc.get(field_name)
            if field_value:
                print(f"   ✅ Field '{field_name}': {len(field_value)} chars")
                print(f"      Preview: {field_value[:200]}...")
            else:
                print(f"   ❌ Field '{field_name}': not found")
    except Exception as e:
        print(f"   ❌ Field '{field_name}': {e}")

## Alternative: Use Index Reader to Get Document Content

In [ ]:
from pyserini.index.lucene import IndexReader

print("Trying IndexReader approach...\n")

index_reader = IndexReader.from_prebuilt_index('robust04')

# Try to get document via index reader
print(f"Getting document {docid} via IndexReader:\n")

# Method 1: get_document_vector
try:
    doc_vector = index_reader.get_document_vector(docid)
    if doc_vector:
        print(f"✅ Document vector: {len(doc_vector)} terms")
        print(f"   Sample terms: {list(doc_vector.keys())[:20]}")
    else:
        print("❌ No document vector")
except Exception as e:
    print(f"❌ get_document_vector failed: {e}")

# Method 2: Try to convert internal docid
try:
    internal_docid = index_reader.convert_collection_docid_to_internal(docid)
    print(f"\n✅ Internal docid: {internal_docid}")
    
    # Now try to get the actual document
    # This might work differently
except Exception as e:
    print(f"\n❌ convert_collection_docid_to_internal failed: {e}")

print("\n" + "="*80)
print("CONCLUSION:")
print("="*80)
print("If no method works to get document content, then neural reranking")
print("is IMPOSSIBLE with this index. We should focus on RM3 optimization instead.")